In [1]:
# Imports
import sklearn
import pandas as pd
import numpy as np

# EDA

In [2]:
dataframe = pd.read_csv('Bitext_Sample_Customer_Support_Training_Dataset_27K_responses-v11.csv', header=0)
print(dataframe.head())
print(dataframe.describe())

   flags                                        instruction category  \
0      B   question about cancelling order {{Order Number}}    ORDER   
1    BQZ  i have a question about cancelling oorder {{Or...    ORDER   
2   BLQZ    i need help cancelling puchase {{Order Number}}    ORDER   
3     BL         I need to cancel purchase {{Order Number}}    ORDER   
4  BCELN  I cannot afford this order, cancel purchase {{...    ORDER   

         intent                                           response  
0  cancel_order  I've understood you have a question regarding ...  
1  cancel_order  I've been informed that you have a question ab...  
2  cancel_order  I can sense that you're seeking assistance wit...  
3  cancel_order  I understood that you need assistance with can...  
4  cancel_order  I'm sensitive to the fact that you're facing f...  
        flags                     instruction category        intent  \
count   26872                           26872    26872         26872   
unique   

In [3]:
# Display unique values in flags, category, and intent columns
# print("Unique values in 'flags' column:")
# print(dataframe['flags'].unique())
# print(f"\nNumber of unique flags: {dataframe['flags'].nunique()}\n")

print("Unique values in 'category' column:")
print(dataframe['category'].unique())
print(f"\nNumber of unique categories: {dataframe['category'].nunique()}\n")

print("Unique values in 'intent' column:")
print(dataframe['intent'].unique())
print(f"\nNumber of unique intents: {dataframe['intent'].nunique()}")

Unique values in 'category' column:
['ORDER' 'SHIPPING' 'CANCEL' 'INVOICE' 'PAYMENT' 'REFUND' 'FEEDBACK'
 'CONTACT' 'ACCOUNT' 'DELIVERY' 'SUBSCRIPTION']

Number of unique categories: 11

Unique values in 'intent' column:
['cancel_order' 'change_order' 'change_shipping_address'
 'check_cancellation_fee' 'check_invoice' 'check_payment_methods'
 'check_refund_policy' 'complaint' 'contact_customer_service'
 'contact_human_agent' 'create_account' 'delete_account'
 'delivery_options' 'delivery_period' 'edit_account' 'get_invoice'
 'get_refund' 'newsletter_subscription' 'payment_issue' 'place_order'
 'recover_password' 'registration_problems' 'review'
 'set_up_shipping_address' 'switch_account' 'track_order' 'track_refund']

Number of unique intents: 27


In [4]:
# Create new dataframe excluding specific intents
intents_to_remove = ["check_invoice", "get_invoice", "payment_issue", "registration_problems", "switch_account", 'edit_account', 'get_refund','newsletter_subscription', 'place_order', 'recover_password', 'review', 'set_up_shipping_address', 'delete_account', 'create_account', 'change_order', 'change_shipping_address', 'cancel_order']
df_filtered = dataframe[~dataframe['intent'].isin(intents_to_remove)]

print(f"Original dataframe shape: {dataframe.shape}")
print(f"Filtered dataframe shape: {df_filtered.shape}")
print(f"Rows removed: {dataframe.shape[0] - df_filtered.shape[0]}")

Original dataframe shape: (26872, 5)
Filtered dataframe shape: (9932, 5)
Rows removed: 16940


In [5]:
# Create label column based on intent mapping
label_mapping = {
    # Map to 1
    'check_cancellation_fee': 1,
    'check_payment_methods': 1,
    'check_refund_policy': 1,
    'delivery_options': 1,
    # Map to -2
    'complaint': -2,
    'contact_customer_service': -2,
    'contact_human_agent': -2,
    # Map to -1
    'delivery_period': -1,
    'track_order': -1,
    'track_refund': -1
}
# Apply mapping to create label column
df_filtered['label'] = df_filtered['intent'].map(label_mapping)

# Check for any unmapped intents
unmapped = df_filtered[df_filtered['label'].isna()]
if len(unmapped) > 0:
    print(f"Warning: {len(unmapped)} rows have unmapped intents:")
    print(unmapped['intent'].unique())
else:
    print("All intents successfully mapped!")

print(f"\nLabel distribution:")
print(df_filtered['label'].value_counts().sort_index())

All intents successfully mapped!

Label distribution:
label
-2    2999
-1    2992
 1    3941
Name: count, dtype: int64


/var/folders/r8/yj4wm4t13l398qfqjr6z16d40000gn/T/ipykernel_2068/3031513557.py:18: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_filtered['label'] = df_filtered['intent'].map(label_mapping)


In [7]:
# Save instructions and label columns to CSV
import csv
df_to_save = df_filtered[['instruction', 'label']].copy()

# Save to CSV with quoting only for non-numeric fields (instruction will be quoted, label won't)
df_to_save.to_csv('More_labeled_data.csv', index=False, quoting=csv.QUOTE_NONNUMERIC)

print(f"Saved {len(df_to_save)} rows to 'More_labeled_data.csv'")
print(f"\nFirst few rows:")
print(df_to_save.head())

Saved 9932 rows to 'DATA.csv'

First few rows:
                                            instruction  label
2968  I can't ifnd the bloody termination charge, I ...      1
2969        what is the fee for canceling the contract?      1
2970               could I check the withdrawal charge?      1
2971   i need help to see the early termination penalty      1
2972          need assistance to see the withdrawal fee      1
